### Bonus tasks

* __2.3 bonus__ (2 pts) Try to find a network architecture and training params that solve __both__ environments above (_Points depend on implementation. If you attempted this task, please mention it in Anytask submission._)

* __2.4 bonus__ (4 pts) Solve continuous action space task with `MLPRegressor` or similar.
  * Since your agent only predicts the "expected" action, you will have to add noise to ensure exploration.
  * Choose one of [MountainCarContinuous-v0](https://gymnasium.farama.org/environments/classic_control/mountain_car_continuous/) (90+ pts to solve), [LunarLanderContinuous-v2](https://gymnasium.farama.org/environments/box2d/lunar_lander/) (`env = gym.make("LunarLander-v2", continuous=True)`)(200+ pts to solve)
  * 4 points for solving. Slightly less for getting some results below solution threshold. Note that discrete and continuous environments may have slightly different rules, aside from action spaces.

## Architectures for solving both tasks
- First attempt is to create some dumb padded network
- Maybe create some small embedder for each task, some general body for both, and decoder to action space

In [116]:
from itertools import chain
import numpy as np
from sklearn.neural_network import MLPClassifier
import gymnasium as gym
import matplotlib.pyplot as plt
from joblib import Parallel, delayed, load, dump

In [117]:
state_dim = 10

In [118]:
def generate_session(env, policy: MLPClassifier, padding, n_actions, t_max=10**4):
    """
    Play game until end or for t_max ticks.
    :param policy: an array of shape [n_states,n_actions] with action probabilities
    :returns: list of states, list of actions and sum of rewards
    """        
    s, _ = env.reset()

    states, actions = [], []
    total_reward = 0.

    for t in range(t_max):
        # force to move
        # FIXME, it's quite bad, that i've used move forcing
        s = np.append(s, padding[:])
        probs = policy.predict_proba(s.reshape(1, -1)).reshape(-1)
        probs_chopped = probs[:n_actions] / np.sum(probs[:n_actions])
        a = np.random.choice(np.arange(n_actions), p=probs_chopped)
        new_s, r, terminated, truncated, _ = env.step(a)

        states.append(s)
        actions.append(a)
        total_reward += r

        s = new_s
        if terminated or truncated:
            break

    return states, actions, total_reward


def run_session(env_id, policy, padding, n_actions, t_max=10**4):
    env = gym.make(env_id, render_mode="rgb_array").env
    states, actions, rewards = generate_session(env, policy, padding, n_actions=n_actions, t_max=t_max)
    env.close()
    return states, actions, rewards

In [119]:
def select_elites(states_batch, actions_batch, rewards_batch, percentile):
    """
    Select states and actions from games that have rewards >= percentile
    :param states_batch: list of lists of states, states_batch[session_i][t]
    :param actions_batch: list of lists of actions, actions_batch[session_i][t]
    :param rewards_batch: list of rewards, rewards_batch[session_i]

    :returns: elite_states,elite_actions, both 1D lists of states and respective actions from elite sessions

    !!
    Please return elite states and actions in their original order
    [i.e. sorted by session number and timestep within session]

    If you are confused, see examples below. Please don't assume that states are integers
    (they will become different later).
    """

    # they say, that selecting elites with same first action is a good idea

    reward_threshold = np.percentile(rewards_batch, percentile)
    mask = rewards_batch > reward_threshold
    elite_states = list(chain.from_iterable([i for i,z in zip(states_batch, mask) if z]))
    elite_actions = list(chain.from_iterable([i for i,z in zip(actions_batch, mask) if z]))

    return elite_states, elite_actions

In [120]:
from IPython.display import clear_output

def show_progress(rewards_batch, log, percentile, reward_range=[-990, +10]):
    """
    A convenience function that displays training progress.
    No cool math here, just charts.
    """

    mean_reward = np.mean(rewards_batch)
    threshold = np.percentile(rewards_batch, percentile)
    log.append([mean_reward, threshold])

    plt.figure(figsize=[8, 4])
    plt.subplot(1, 2, 1)
    plt.plot(list(zip(*log))[0], label="Mean rewards")
    plt.plot(list(zip(*log))[1], label="Reward thresholds")
    plt.legend()
    plt.grid()

    plt.subplot(1, 2, 2)
    plt.hist(rewards_batch, range=reward_range)
    plt.vlines(
        [np.percentile(rewards_batch, percentile)],
        [0],
        [100],
        label="percentile",
        color="red",
    )
    plt.legend()
    plt.grid()
    print("mean reward = %.3f, threshold=%.3f" % (mean_reward, threshold))
    plt.show()

In [121]:
def train(n_epochs, n_sessions, percentile, agent):
    log = []

    memory = 5
    previous_sessions = []

    for i in range(n_epochs):
        clear_output(True)
        for id, n_actions, n_states in zip(["MountainCar-v0", 'LunarLander-v2'], [3, 4], [2, 8]):
            # pad
            padding = [0] * (state_dim - n_states)
            padding[6] = id == 'MountainCar-v0'
            padding[7] = id == 'LunarLander-v2'

            sessions = Parallel(n_jobs=1)(
                delayed(run_session)(id, agent, padding, n_actions=n_actions, t_max=700)
                for _ in range(n_sessions)
            )
            states, actions, rewards = zip(*sessions)

            states = [i + padding[:] for i in states]

            if len(previous_sessions) == memory:
                for j in range(memory - 1):
                    previous_sessions[j] = previous_sessions[j + 1]
                previous_sessions[j + 1] = (states, actions, rewards)
            else:
                previous_sessions.append((states, actions, rewards))

            states, actions, rewards = zip(*chain.from_iterable(zip(*i) for i in previous_sessions))

            elite_states, elite_actions = select_elites(states, actions, rewards, percentile)
            agent.partial_fit(elite_states, elite_actions, classes=range(10))
            
            # all rewards, not only elite
            show_progress(rewards, log, percentile)
            print(f"Epoch {i+1}")


agent = MLPClassifier(
    (128, 64, 32, 16),
    'relu'
)
# 8 states for Lunar Lander + 2 dim one-hot for task
# I think I will put velocities into such of other agent
# maybe it's dumb
# it's dumb I think, but let's try
agent.partial_fit(np.arange(state_dim).reshape(1, -1), [0.], classes=np.arange(4))

n_epochs = 100
n_sessions = 250
percentile = 30
# train(n_epochs, n_sessions, percentile, agent)
train(n_epochs, n_sessions, percentile, agent)

KeyboardInterrupt: 